In [1]:
from pathlib import Path
import pandas as pd

files = [
    "../data/interim/plant1_forecasting_dataset.csv",
    "../data/interim/plant2_forecasting_dataset.csv",
    "../data/processed/splits/plant1_train.csv",
    "../data/processed/splits/plant1_validation.csv",
    "../data/processed/splits/plant1_test.csv",
    "../data/processed/splits/plant2_train.csv",
    "../data/processed/splits/plant2_validation.csv",
    "../data/processed/splits/plant2_test.csv",
]

for file in files:
    df = pd.read_csv(file, parse_dates=["DATE_TIME"])

    print(f"\n{file}")
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print("Dates:", df["DATE_TIME"].min(), "to", df["DATE_TIME"].max())

    if "TARGET_AC_POWER_15MIN" in df.columns:
        print("Rows with a future target:",
              df["TARGET_AC_POWER_15MIN"].notna().sum())


../data/interim/plant1_forecasting_dataset.csv
Rows: 3158
Columns: ['PLANT_ID', 'DATE_TIME', 'AC_POWER', 'DC_POWER', 'source_count', 'expected_sources', 'complete_coverage', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION', 'TARGET_AC_POWER_15MIN', 'TARGET_TIMESTAMP', 'HAS_EXACT_FUTURE_TIMESTAMP']
Dates: 2020-05-15 00:00:00 to 2020-06-17 23:45:00
Rows with a future target: 3148

../data/interim/plant2_forecasting_dataset.csv
Rows: 3259
Columns: ['PLANT_ID', 'DATE_TIME', 'AC_POWER', 'DC_POWER', 'source_count', 'expected_sources', 'complete_coverage', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION', 'TARGET_AC_POWER_15MIN', 'TARGET_TIMESTAMP', 'HAS_EXACT_FUTURE_TIMESTAMP']
Dates: 2020-05-15 00:00:00 to 2020-06-17 23:45:00
Rows with a future target: 3253

../data/processed/splits/plant1_train.csv
Rows: 2202
Columns: ['PLANT_ID', 'DATE_TIME', 'AC_POWER', 'DC_POWER', 'source_count', 'expected_sources', 'complete_coverage', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', '

In [2]:
from pathlib import Path

print("Notebook is looking from:", Path.cwd())
print("data/interim exists here:", Path("data/interim").exists())
print("../data/interim exists here:", Path("../data/interim").exists())

Notebook is looking from: /Users/apple/Documents/MLOps_EL/SolarSelect/EDA
data/interim exists here: False
../data/interim exists here: True


In [3]:
import pandas as pd

for plant in ["plant1", "plant2"]:
    file = f"../data/interim/{plant}_forecasting_dataset.csv"
    df = pd.read_csv(file, parse_dates=["DATE_TIME", "TARGET_TIMESTAMP"])

    power_at_time = df.set_index("DATE_TIME")["AC_POWER"]
    expected_target = (
        df["DATE_TIME"] + pd.Timedelta(minutes=15)
    ).map(power_at_time)

    has_target = df["TARGET_AC_POWER_15MIN"].notna()
    target_matches = (
        has_target
        & expected_target.notna()
        & (
            df["TARGET_AC_POWER_15MIN"]
            == expected_target
        )
    )

    print(f"\n{plant}")
    print("Rows with a target:", has_target.sum())
    print("Targets matching power 15 minutes later:", target_matches.sum())
    print("Target timestamps exactly 15 minutes later:",
          (df["TARGET_TIMESTAMP"] ==
           df["DATE_TIME"] + pd.Timedelta(minutes=15)).sum())


plant1
Rows with a target: 3148
Targets matching power 15 minutes later: 3148
Target timestamps exactly 15 minutes later: 3158

plant2
Rows with a target: 3253
Targets matching power 15 minutes later: 3253
Target timestamps exactly 15 minutes later: 3259


In [4]:
import pandas as pd

for plant in ["plant1", "plant2"]:
    print(f"\n{plant}")

    for earlier, later in [
        ("train", "validation"),
        ("validation", "test"),
    ]:
        earlier_file = f"../data/processed/splits/{plant}_{earlier}.csv"
        later_file = f"../data/processed/splits/{plant}_{later}.csv"

        earlier_df = pd.read_csv(
            earlier_file, parse_dates=["TARGET_TIMESTAMP"]
        )
        later_df = pd.read_csv(
            later_file, parse_dates=["DATE_TIME"]
        )

        last_target = earlier_df["TARGET_TIMESTAMP"].max()
        next_period_start = later_df["DATE_TIME"].min()

        print(
            f"{earlier} last target: {last_target} | "
            f"{later} starts: {next_period_start} | "
            f"crosses into next period: {last_target >= next_period_start}"
        )


plant1
train last target: 2020-06-08 02:30:00 | validation starts: 2020-06-08 02:45:00 | crosses into next period: False
validation last target: 2020-06-13 00:30:00 | test starts: 2020-06-13 00:45:00 | crosses into next period: False

plant2
train last target: 2020-06-07 19:30:00 | validation starts: 2020-06-07 19:45:00 | crosses into next period: False
validation last target: 2020-06-12 21:30:00 | test starts: 2020-06-12 21:45:00 | crosses into next period: False


In [5]:
import numpy as np
import pandas as pd

for plant in ["plant1", "plant2"]:
    file = f"../data/processed/splits/{plant}_validation.csv"
    df = pd.read_csv(file).dropna(
        subset=["TARGET_AC_POWER_15MIN", "AC_POWER", "IRRADIATION"]
    )

    actual = df["TARGET_AC_POWER_15MIN"].to_numpy()
    predicted = df["AC_POWER"].to_numpy()
    error = predicted - actual

    mae = np.mean(np.abs(error))
    rmse = np.sqrt(np.mean(error ** 2))
    r2 = 1 - np.sum(error ** 2) / np.sum((actual - actual.mean()) ** 2)
    bias = np.mean(error)

    daytime = df["IRRADIATION"].to_numpy() > 0
    daytime_error = error[daytime]

    print(f"\n{plant} — validation")
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R²:", r2)
    print("Daytime MAE:", np.mean(np.abs(daytime_error)))
    print("Daytime RMSE:", np.sqrt(np.mean(daytime_error ** 2)))
    print("Bias (positive means predictions are too high):", bias)


plant1 — validation
MAE: 1112.471671219066
RMSE: 2095.357793880018
R²: 0.9336115667504671
Daytime MAE: 1992.297175453156
Daytime RMSE: 2804.08239733192
Bias (positive means predictions are too high): 7.723946511872002e-15

plant2 — validation
MAE: 1052.0631357417394
RMSE: 2130.59889992422
R²: 0.8708641262124635
Daytime MAE: 1823.3265021573918
Daytime RMSE: 2804.871819400781
Bias (positive means predictions are too high): -1.8675455888561155e-15


In [6]:
import importlib.util

print("XGBoost available:", importlib.util.find_spec("xgboost") is not None)

XGBoost available: True


In [7]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [8]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

def make_features(df):
    features = df[
        [
            "AC_POWER",
            "DC_POWER",
            "AMBIENT_TEMPERATURE",
            "MODULE_TEMPERATURE",
            "IRRADIATION",
        ]
    ].copy()

    time = pd.to_datetime(df["DATE_TIME"])
    hour = time.dt.hour + time.dt.minute / 60

    features["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    features["hour_cos"] = np.cos(2 * np.pi * hour / 24)

    return features

for plant in ["plant1", "plant2"]:
    train = pd.read_csv(
        f"../data/processed/splits/{plant}_train.csv"
    )
    validation = pd.read_csv(
        f"../data/processed/splits/{plant}_validation.csv"
    )

    model = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )

    model.fit(
        make_features(train),
        train["TARGET_AC_POWER_15MIN"],
    )

    actual = validation["TARGET_AC_POWER_15MIN"].to_numpy()
    predicted = model.predict(make_features(validation))
    error = predicted - actual

    daytime = validation["IRRADIATION"].to_numpy() > 0
    daytime_error = error[daytime]

    r2 = 1 - np.sum(error ** 2) / np.sum(
        (actual - actual.mean()) ** 2
    )

    print(f"\n{plant} — XGBoost validation")
    print("MAE:", np.mean(np.abs(error)))
    print("RMSE:", np.sqrt(np.mean(error ** 2)))
    print("R²:", r2)
    print("Daytime MAE:", np.mean(np.abs(daytime_error)))
    print("Daytime RMSE:", np.sqrt(np.mean(daytime_error ** 2)))
    print("Bias (positive means predictions are too high):",
          np.mean(error))


plant1 — XGBoost validation
MAE: 989.1091400641966
RMSE: 1972.5412223948492
R²: 0.9411660222747497
Daytime MAE: 1755.7433471002357
Daytime RMSE: 2639.5618659355864
Bias (positive means predictions are too high): 166.97581338690324

plant2 — XGBoost validation
MAE: 922.7390823049277
RMSE: 1978.4755326605427
R²: 0.8886462425328515
Daytime MAE: 1586.69120745698
Daytime RMSE: 2604.5289942556756
Bias (positive means predictions are too high): 153.4610922961326


In [9]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [10]:
import importlib.util
import sys

print("Notebook Python:", sys.executable)
print("scikit-learn found:", importlib.util.find_spec("sklearn") is not None)

Notebook Python: /opt/anaconda3/envs/solarselect/bin/python
scikit-learn found: True


In [11]:
%pip install mlflow

  Using cached annotated_doc-0.0.5-py3-none-any.whl.metadata (6.5 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 42.1 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 47.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 42.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 31.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 50.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.3 MB/s  0:00:00
Using cached annotated_doc-0.0.5-py3-none-any.whl (5.3 kB)
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)
Using cached itsdangerous-2.2.0-py3-none-any.whl (16 kB)
Using cached pyasn1_modules-0.4.2-py3-none-any.whl (181 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━

In [12]:
from pathlib import Path
import mlflow

project_root = Path.cwd().parent
tracking_folder = project_root / "mlruns"

mlflow.set_tracking_uri(tracking_folder.as_uri())
mlflow.set_experiment("SolarSelect-Forecasting")

print("Saving MLflow records in:", tracking_folder)
print("Experiment:", mlflow.get_experiment_by_name(
    "SolarSelect-Forecasting"
).name)

MlflowException: The filesystem tracking backend (e.g., './mlruns') is in maintenance mode and will not receive further updates. Please migrate to a database backend (e.g., 'sqlite:///mlflow.db') to access the latest MLflow features. The `mlflow migrate-filestore` tool migrates your existing data losslessly. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance. If the filesystem backend is required for your workflow, set `MLFLOW_ALLOW_FILE_STORE=true` to opt out of this exception.

In [13]:
from pathlib import Path
import mlflow

project_root = Path.cwd().parent
tracking_db = project_root / "mlflow.db"

mlflow.set_tracking_uri(f"sqlite:///{tracking_db.as_posix()}")
experiment = mlflow.set_experiment("SolarSelect-Forecasting")

print("Tracking database:", tracking_db)
print("Experiment:", experiment.name)
print("Experiment ID:", experiment.experiment_id)
print("Artifact location:", experiment.artifact_location)

2026/10/13 22:10:59 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/13 22:10:59 INFO mlflow.store.db.utils: Updating database tables
2026/10/13 22:11:00 INFO mlflow.tracking.fluent: Experiment with name 'SolarSelect-Forecasting' does not exist. Creating a new experiment.


Tracking database: /Users/apple/Documents/MLOps_EL/SolarSelect/mlflow.db
Experiment: SolarSelect-Forecasting
Experiment ID: 1
Artifact location: /Users/apple/Documents/MLOps_EL/SolarSelect/EDA/mlruns/1


In [14]:
from pathlib import Path
import mlflow
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

# Reconnect this notebook to the local MLflow experiment
project_root = Path.cwd().parent
tracking_db = project_root / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{tracking_db.as_posix()}")
mlflow.set_experiment("SolarSelect-Forecasting")


def make_features(df):
    features = df[
        [
            "AC_POWER",
            "DC_POWER",
            "AMBIENT_TEMPERATURE",
            "MODULE_TEMPERATURE",
            "IRRADIATION",
        ]
    ].copy()

    time = pd.to_datetime(df["DATE_TIME"])
    hour = time.dt.hour + time.dt.minute / 60
    features["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    features["hour_cos"] = np.cos(2 * np.pi * hour / 24)

    return features


def calculate_scores(df, predictions):
    actual = df["TARGET_AC_POWER_15MIN"].to_numpy()
    error = predictions - actual
    daytime = df["IRRADIATION"].to_numpy() > 0
    daytime_error = error[daytime]

    return {
        "mae": np.mean(np.abs(error)),
        "rmse": np.sqrt(np.mean(error ** 2)),
        "r2": 1 - np.sum(error ** 2) / np.sum(
            (actual - actual.mean()) ** 2
        ),
        "daytime_mae": np.mean(np.abs(daytime_error)),
        "daytime_rmse": np.sqrt(np.mean(daytime_error ** 2)),
        "bias": np.mean(error),
    }


for plant in ["plant1", "plant2"]:
    train = pd.read_csv(
        f"../data/processed/splits/{plant}_train.csv"
    )
    validation = pd.read_csv(
        f"../data/processed/splits/{plant}_validation.csv"
    )

    # Record the persistence baseline
    baseline_predictions = validation["AC_POWER"].to_numpy()
    baseline_scores = calculate_scores(validation, baseline_predictions)

    with mlflow.start_run(run_name=f"persistence-{plant}"):
        mlflow.log_params({
            "model": "persistence",
            "plant": plant,
            "prediction_rule": "current AC_POWER",
        })
        mlflow.log_metrics(baseline_scores)

    print(f"{plant} persistence logged:", baseline_scores)

    # Train and record XGBoost
    model = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(
        make_features(train),
        train["TARGET_AC_POWER_15MIN"],
    )

    xgb_predictions = model.predict(make_features(validation))
    xgb_scores = calculate_scores(validation, xgb_predictions)

    with mlflow.start_run(run_name=f"xgboost-{plant}"):
        mlflow.log_params({
            "model": "xgboost",
            "plant": plant,
            "n_estimators": 300,
            "max_depth": 4,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "features": "current power, current weather, time of day",
        })
        mlflow.log_metrics(xgb_scores)
        mlflow.xgboost.log_model(model, name="model")

    print(f"{plant} XGBoost logged:", xgb_scores)

plant1 persistence logged: {'mae': np.float64(1112.471671219066), 'rmse': np.float64(2095.357793880018), 'r2': np.float64(0.9336115667504671), 'daytime_mae': np.float64(1992.297175453156), 'daytime_rmse': np.float64(2804.08239733192), 'bias': np.float64(7.723946511872002e-15)}
plant1 XGBoost logged: {'mae': np.float64(989.1091400641966), 'rmse': np.float64(1972.5412223948492), 'r2': np.float64(0.9411660222747497), 'daytime_mae': np.float64(1755.7433471002357), 'daytime_rmse': np.float64(2639.5618659355864), 'bias': np.float64(166.97581338690324)}
plant2 persistence logged: {'mae': np.float64(1052.0631357417394), 'rmse': np.float64(2130.59889992422), 'r2': np.float64(0.8708641262124635), 'daytime_mae': np.float64(1823.3265021573918), 'daytime_rmse': np.float64(2804.871819400781), 'bias': np.float64(-1.8675455888561155e-15)}
plant2 XGBoost logged: {'mae': np.float64(922.7390823049277), 'rmse': np.float64(1978.4755326605427), 'r2': np.float64(0.8886462425328515), 'daytime_mae': np.float64

In [15]:
experiment = mlflow.get_experiment_by_name(
    "SolarSelect-Forecasting"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

runs[
    [
        "tags.mlflow.runName",
        "params.model",
        "params.plant",
        "metrics.mae",
        "metrics.rmse",
        "metrics.r2",
        "metrics.daytime_mae",
        "metrics.daytime_rmse",
        "metrics.bias",
    ]
]

,tags.mlflow.runName,params.model,params.plant,metrics.mae,metrics.rmse,metrics.r2,metrics.daytime_mae,metrics.daytime_rmse,metrics.bias
0,xgboost-plant2,xgboost,plant2,922.739082,1978.475533,0.888646,1586.691207,2604.528994,1.534611e+02
1,persistence-plant2,persistence,plant2,1052.063136,2130.598900,0.870864,1823.326502,2804.871819,-1.867546e-15
2,xgboost-plant1,xgboost,plant1,989.109140,1972.541222,0.941166,1755.743347,2639.561866,1.669758e+02
3,persistence-plant1,persistence,plant1,1112.471671,2095.357794,0.933612,1992.297175,2804.082397,7.723947e-15


In [16]:
from pathlib import Path
import mlflow
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

project_root = Path.cwd().parent
tracking_db = project_root / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{tracking_db.as_posix()}")
mlflow.set_experiment("SolarSelect-Forecasting")


def make_features(df):
    features = df[
        [
            "AC_POWER",
            "DC_POWER",
            "AMBIENT_TEMPERATURE",
            "MODULE_TEMPERATURE",
            "IRRADIATION",
        ]
    ].copy()

    time = pd.to_datetime(df["DATE_TIME"])
    hour = time.dt.hour + time.dt.minute / 60
    features["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    features["hour_cos"] = np.cos(2 * np.pi * hour / 24)

    return features


def calculate_scores(df, predictions):
    actual = df["TARGET_AC_POWER_15MIN"].to_numpy()
    error = predictions - actual
    daytime = df["IRRADIATION"].to_numpy() > 0
    daytime_error = error[daytime]

    return {
        "mae": np.mean(np.abs(error)),
        "rmse": np.sqrt(np.mean(error ** 2)),
        "r2": 1 - np.sum(error ** 2) / np.sum(
            (actual - actual.mean()) ** 2
        ),
        "daytime_mae": np.mean(np.abs(daytime_error)),
        "daytime_rmse": np.sqrt(np.mean(daytime_error ** 2)),
        "bias": np.mean(error),
    }


for plant in ["plant1", "plant2"]:
    train = pd.read_csv(
        f"../data/processed/splits/{plant}_train.csv"
    )
    validation = pd.read_csv(
        f"../data/processed/splits/{plant}_validation.csv"
    )
    test = pd.read_csv(
        f"../data/processed/splits/{plant}_test.csv"
    )

    train_and_validation = pd.concat(
        [train, validation],
        ignore_index=True,
    )

    # Final persistence score on test
    baseline_predictions = test["AC_POWER"].to_numpy()
    baseline_scores = calculate_scores(test, baseline_predictions)

    with mlflow.start_run(run_name=f"final-test-persistence-{plant}"):
        mlflow.log_params({
            "model": "persistence",
            "plant": plant,
            "evaluation_split": "test",
        })
        mlflow.log_metrics(baseline_scores)

    print(f"\n{plant} — final persistence test scores")
    print(baseline_scores)

    # Final XGBoost model: learn from train + validation, score on test
    model = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(
        make_features(train_and_validation),
        train_and_validation["TARGET_AC_POWER_15MIN"],
    )

    xgb_predictions = model.predict(make_features(test))
    xgb_scores = calculate_scores(test, xgb_predictions)

    with mlflow.start_run(run_name=f"final-test-xgboost-{plant}"):
        mlflow.log_params({
            "model": "xgboost",
            "plant": plant,
            "evaluation_split": "test",
            "trained_on": "train plus validation",
            "n_estimators": 300,
            "max_depth": 4,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
        })
        mlflow.log_metrics(xgb_scores)
        mlflow.xgboost.log_model(model, name="model")

    print(f"{plant} — final XGBoost test scores")
    print(xgb_scores)


plant1 — final persistence test scores
{'mae': np.float64(1219.4393423471586), 'rmse': np.float64(2327.4456344528126), 'r2': np.float64(0.9199834171909287), 'daytime_mae': np.float64(2201.484010366107), 'daytime_rmse': np.float64(3127.2274313844878), 'bias': np.float64(1.8419699486553933)}
plant1 — final XGBoost test scores
{'mae': np.float64(1156.5595335563362), 'rmse': np.float64(2312.6517776370165), 'r2': np.float64(0.9209973972607339), 'daytime_mae': np.float64(2073.0259231114264), 'daytime_rmse': np.float64(3107.2544328137656), 'bias': np.float64(128.00205571621836)}

plant2 — final persistence test scores
{'mae': np.float64(1215.689069086651), 'rmse': np.float64(2349.618280208466), 'r2': np.float64(0.8262204857904358), 'daytime_mae': np.float64(2173.0998744113026), 'daytime_rmse': np.float64(3141.41924401052), 'bias': np.float64(1.863718651174033e-14)}
plant2 — final XGBoost test scores
{'mae': np.float64(1099.077887542252), 'rmse': np.float64(2175.6486191611657), 'r2': np.float

In [17]:
experiment = mlflow.get_experiment_by_name(
    "SolarSelect-Forecasting"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

final_runs = runs[
    runs["tags.mlflow.runName"].str.startswith("final-test-")
]

final_runs[
    [
        "tags.mlflow.runName",
        "metrics.mae",
        "metrics.rmse",
        "metrics.r2",
        "metrics.daytime_mae",
        "metrics.daytime_rmse",
        "metrics.bias",
    ]
]

,tags.mlflow.runName,metrics.mae,metrics.rmse,metrics.r2,metrics.daytime_mae,metrics.daytime_rmse,metrics.bias
0,final-test-xgboost-plant2,1099.077888,2175.648619,0.851002,1955.904016,2908.785803,3.811950e+01
1,final-test-persistence-plant2,1215.689069,2349.618280,0.826220,2173.099874,3141.419244,1.863719e-14
2,final-test-xgboost-plant1,1156.559534,2312.651778,0.920997,2073.025923,3107.254433,1.280021e+02
3,final-test-persistence-plant1,1219.439342,2327.445634,0.919983,2201.484010,3127.227431,1.841970e+00
